# STAGE 5.3 — HAND ROI EXTRACTION

In [2]:
# =============================================================================
# PARASURG / CISLR
# STAGE 5.3 — HAND ROI EXTRACTION
#
# FINAL CLEAN VERSION
#
# IMPORTANT:
#   - This cell replaces ALL previous Stage 5.3 extraction/repair cells.
#   - It deletes ONLY previous Stage 5.3 outputs.
#   - Stage 5.1 and Stage 5.2 are untouched.
#   - Raw CISLR videos are NEVER modified.
#   - Stage 4 artifacts are NEVER modified.
#   - No training.
#   - No retrieval.
#
# DECODING STRATEGY
# -----------------
# 1. OpenCV direct/sequential decoding is attempted.
# 2. If OpenCV cannot recover the requested frames, FFmpeg is used.
# 3. FFmpeg decodes the video sequentially and exact requested frame indices
#    are retained.
# 4. No neighboring-frame substitution is allowed.
#
# SCIENTIFIC RULE
# ---------------
# Requested frame index == recovered frame index.
# If an exact frame cannot be decoded, it is recorded as a genuine failure.
# =============================================================================


from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import subprocess
import shutil
import hashlib
import json
import time
import re
import os
import sys


# =============================================================================
# 0. ROOT PATHS
# =============================================================================

ROOT = Path(
    "/home/dipshikha/Music/cao"
)

RESEARCH_ROOT = (
    ROOT / "CISLR_RESEARCH"
)

STAGE5_ROOT = (
    RESEARCH_ROOT
    / "audit"
    / "stage5_hand_roi"
)

MASTER_MANIFEST = (
    STAGE5_ROOT
    / "manifests"
    / "stage5_1_master_video_manifest.csv"
)


# =============================================================================
# 1. STAGE 5.3 OUTPUT DIRECTORIES
#
# ONLY THESE ARE DELETED.
# =============================================================================

ROI_EXTRACTION = (
    STAGE5_ROOT
    / "roi_extraction"
)

ROI_DATASET = (
    STAGE5_ROOT
    / "roi_dataset"
)


# =============================================================================
# 2. CLEAN PREVIOUS STAGE 5.3 OUTPUTS
# =============================================================================

print("=" * 80)
print("STAGE 5.3 — CLEAN RESTART")
print("=" * 80)

print("\nThe following previous Stage 5.3 outputs will be removed:")

print(
    f"  {ROI_EXTRACTION}"
)

print(
    f"  {ROI_DATASET}"
)

print(
    "\nStage 5.1 and Stage 5.2 will NOT be touched."
)

print(
    "Raw CISLR videos will NOT be touched."
)

print(
    "Stage 4 artifacts will NOT be touched."
)


for directory in [
    ROI_EXTRACTION,
    ROI_DATASET
]:

    if directory.exists():

        print(
            f"\nRemoving: {directory}"
        )

        shutil.rmtree(
            directory
        )

    else:

        print(
            f"\nNot present: {directory}"
        )


ROI_EXTRACTION.mkdir(
    parents=True,
    exist_ok=True
)

ROI_DATASET.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. OUTPUT FILES
# =============================================================================

FRAME_AUDIT = (
    ROI_EXTRACTION
    / "stage5_3_frame_level_extraction.csv"
)

VIDEO_AUDIT = (
    ROI_EXTRACTION
    / "stage5_3_video_level_summary.csv"
)

ERROR_AUDIT = (
    ROI_EXTRACTION
    / "stage5_3_extraction_errors.csv"
)

SUMMARY_FILE = (
    ROI_EXTRACTION
    / "stage5_3_extraction_summary.json"
)

HASH_FILE = (
    ROI_EXTRACTION
    / "stage5_3_input_output_hashes.json"
)

PROTOCOL_FILE = (
    ROI_EXTRACTION
    / "stage5_3_extraction_protocol.json"
)


# =============================================================================
# 4. FROZEN PARAMETERS
# =============================================================================

EXPECTED_FRAMES_PER_VIDEO = 8

OUTPUT_SIZE = 224

# Stage 5.2 selected detector
DETECTOR_NAME = (
    "OpenCV_Skin_Contour"
)

# Stage 5.2 detector parameters
HSV_LOWER = np.array(
    [0, 20, 40],
    dtype=np.uint8
)

HSV_UPPER = np.array(
    [35, 255, 255],
    dtype=np.uint8
)

YCRCB_LOWER = np.array(
    [0, 133, 77],
    dtype=np.uint8
)

YCRCB_UPPER = np.array(
    [255, 173, 127],
    dtype=np.uint8
)

MORPH_KERNEL_SIZE = 5

MIN_CONTOUR_AREA = 150

ROI_MIN_AREA_FRACTION = 0.005

ROI_MAX_AREA_FRACTION = 0.90

CROP_PADDING_FACTOR = 0.35

JPEG_QUALITY = 95


# =============================================================================
# 5. CHECK FFMPEG
# =============================================================================

print("\n" + "=" * 80)
print("DECODER AVAILABILITY")
print("=" * 80)

FFMPEG_BIN = shutil.which(
    "ffmpeg"
)

FFPROBE_BIN = shutil.which(
    "ffprobe"
)

print(
    "ffmpeg :",
    FFMPEG_BIN
)

print(
    "ffprobe:",
    FFPROBE_BIN
)

if FFMPEG_BIN is None:

    raise RuntimeError(
        "\nFFmpeg is required for the final robust Stage 5.3 "
        "decoder fallback, but it was not found in PATH.\n"
        "Install it with:\n"
        "sudo apt install ffmpeg"
    )


# =============================================================================
# 6. HELPERS
# =============================================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def safe_uid_folder(uid):

    uid = str(uid)

    safe = re.sub(
        r"[^A-Za-z0-9_.=-]",
        "_",
        uid
    )

    digest = hashlib.sha256(
        uid.encode(
            "utf-8"
        )
    ).hexdigest()[:8]

    return (
        f"{safe}_{digest}"
    )


def detect_opencv_skin_contour(
    frame_bgr
):

    h, w = frame_bgr.shape[:2]

    frame_area = float(
        w * h
    )

    hsv = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2HSV
    )

    ycrcb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2YCrCb
    )

    hsv_mask = cv2.inRange(
        hsv,
        HSV_LOWER,
        HSV_UPPER
    )

    ycrcb_mask = cv2.inRange(
        ycrcb,
        YCRCB_LOWER,
        YCRCB_UPPER
    )

    mask = cv2.bitwise_and(
        hsv_mask,
        ycrcb_mask
    )

    kernel = np.ones(
        (
            MORPH_KERNEL_SIZE,
            MORPH_KERNEL_SIZE
        ),
        dtype=np.uint8
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    candidates = []

    for contour in contours:

        area = cv2.contourArea(
            contour
        )

        if area < MIN_CONTOUR_AREA:
            continue

        x, y, bw, bh = cv2.boundingRect(
            contour
        )

        if bw <= 0 or bh <= 0:
            continue

        bbox_area_fraction = (
            float(bw * bh)
            / frame_area
        )

        if (
            bbox_area_fraction
            < ROI_MIN_AREA_FRACTION
        ):
            continue

        if (
            bbox_area_fraction
            > ROI_MAX_AREA_FRACTION
        ):
            continue

        candidates.append({
            "area": float(area),
            "x1": int(x),
            "y1": int(y),
            "x2": int(x + bw),
            "y2": int(y + bh)
        })

    if not candidates:

        return (
            None,
            0.0,
            0
        )

    candidates = sorted(
        candidates,
        key=lambda x: x["area"],
        reverse=True
    )[:2]

    x1 = min(
        c["x1"]
        for c in candidates
    )

    y1 = min(
        c["y1"]
        for c in candidates
    )

    x2 = max(
        c["x2"]
        for c in candidates
    )

    y2 = max(
        c["y2"]
        for c in candidates
    )

    merged_area_fraction = (
        float(
            (x2 - x1)
            * (y2 - y1)
        )
        / frame_area
    )

    if (
        merged_area_fraction
        < ROI_MIN_AREA_FRACTION
        or
        merged_area_fraction
        > ROI_MAX_AREA_FRACTION
    ):

        return (
            None,
            0.0,
            len(candidates)
        )

    contour_area_sum = sum(
        c["area"]
        for c in candidates
    )

    confidence_proxy = min(
        contour_area_sum
        / frame_area,
        1.0
    )

    return (
        (
            x1,
            y1,
            x2,
            y2
        ),
        float(
            confidence_proxy
        ),
        len(candidates)
    )


def crop_square_with_padding(
    frame_bgr,
    roi_box
):

    h, w = frame_bgr.shape[:2]

    x1, y1, x2, y2 = roi_box

    bw = max(
        1,
        x2 - x1
    )

    bh = max(
        1,
        y2 - y1
    )

    cx = (
        x1 + x2
    ) / 2.0

    cy = (
        y1 + y2
    ) / 2.0

    side = max(
        bw,
        bh
    )

    side *= (
        1.0
        + CROP_PADDING_FACTOR
    )

    crop_x1 = int(
        round(
            cx - side / 2
        )
    )

    crop_y1 = int(
        round(
            cy - side / 2
        )
    )

    crop_x2 = int(
        round(
            cx + side / 2
        )
    )

    crop_y2 = int(
        round(
            cy + side / 2
        )
    )

    if crop_x1 < 0:

        shift = -crop_x1

        crop_x1 += shift
        crop_x2 += shift

    if crop_y1 < 0:

        shift = -crop_y1

        crop_y1 += shift
        crop_y2 += shift

    if crop_x2 > w:

        shift = crop_x2 - w

        crop_x1 -= shift
        crop_x2 -= shift

    if crop_y2 > h:

        shift = crop_y2 - h

        crop_y1 -= shift
        crop_y2 -= shift

    crop_x1 = max(
        0,
        crop_x1
    )

    crop_y1 = max(
        0,
        crop_y1
    )

    crop_x2 = min(
        w,
        crop_x2
    )

    crop_y2 = min(
        h,
        crop_y2
    )

    if (
        crop_x2 <= crop_x1
        or
        crop_y2 <= crop_y1
    ):

        return (
            None,
            None
        )

    crop = frame_bgr[
        crop_y1:crop_y2,
        crop_x1:crop_x2
    ]

    if crop.size == 0:

        return (
            None,
            None
        )

    crop_resized = cv2.resize(
        crop,
        (
            OUTPUT_SIZE,
            OUTPUT_SIZE
        ),
        interpolation=cv2.INTER_AREA
    )

    return (
        crop_resized,
        (
            crop_x1,
            crop_y1,
            crop_x2,
            crop_y2
        )
    )


# =============================================================================
# 7. ROBUST VIDEO FRAME DECODER
# =============================================================================
#
# This function attempts:
#
#   A. OpenCV sequential decoding
#   B. FFmpeg sequential decoding
#
# It returns ONLY exact requested frame indices.
# =============================================================================

def decode_requested_frames(
    video_path,
    requested_indices
):

    requested_indices = sorted(
        set(
            int(x)
            for x in requested_indices
        )
    )

    if not requested_indices:

        return {}, {
            "decoder": "none",
            "error": None
        }

    max_target = max(
        requested_indices
    )

    requested_set = set(
        requested_indices
    )

    # -------------------------------------------------------------------------
    # A. OpenCV sequential decoding
    # -------------------------------------------------------------------------

    frames_found = {}

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if cap.isOpened():

        idx = 0

        while idx <= max_target:

            ok, frame = cap.read()

            if not ok or frame is None:

                break

            if idx in requested_set:

                frames_found[idx] = (
                    frame.copy()
                )

            idx += 1

        cap.release()

    else:

        cap.release()

    if len(frames_found) == len(
        requested_set
    ):

        return frames_found, {
            "decoder":
                "OpenCV_sequential",
            "error": None
        }

    # -------------------------------------------------------------------------
    # B. FFmpeg rawvideo fallback
    # -------------------------------------------------------------------------
    #
    # Force FFmpeg to decode the entire stream sequentially.
    # Output is raw BGR24 through stdout.
    #
    # This avoids OpenCV's frame-seeking/indexing behavior.
    # -------------------------------------------------------------------------

    frames_found = {}

    # Probe dimensions first
    probe_cmd = [
        FFPROBE_BIN,
        "-v",
        "error",
        "-select_streams",
        "v:0",
        "-show_entries",
        "stream=width,height",
        "-of",
        "csv=p=0:s=x",
        str(video_path)
    ]

    try:

        probe = subprocess.run(
            probe_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=60
        )

        if probe.returncode != 0:

            return {}, {
                "decoder":
                    "FFmpeg_probe_failed",
                "error":
                    probe.stderr.strip()
            }

        dimension_text = (
            probe.stdout.strip()
        )

        if "x" not in dimension_text:

            return {}, {
                "decoder":
                    "FFmpeg_probe_failed",
                "error":
                    "Could not determine video dimensions"
            }

        width, height = (
            map(
                int,
                dimension_text.split("x")
            )
        )

    except Exception as exc:

        return {}, {
            "decoder":
                "FFmpeg_probe_exception",
            "error":
                str(exc)
        }

    frame_bytes = (
        width
        * height
        * 3
    )

    ffmpeg_cmd = [
        FFMPEG_BIN,

        "-hide_banner",

        "-loglevel",
        "error",

        "-i",
        str(video_path),

        "-map",
        "0:v:0",

        "-f",
        "rawvideo",

        "-pix_fmt",
        "bgr24",

        "-"
    ]

    try:

        process = subprocess.Popen(
            ffmpeg_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE
        )

        idx = 0

        while idx <= max_target:

            raw = process.stdout.read(
                frame_bytes
            )

            if len(raw) != frame_bytes:

                break

            if idx in requested_set:

                frame = np.frombuffer(
                    raw,
                    dtype=np.uint8
                ).reshape(
                    (
                        height,
                        width,
                        3
                    )
                ).copy()

                frames_found[idx] = frame

            idx += 1

        stdout_remaining, stderr_data = (
            process.communicate(
                timeout=120
            )
        )

        if (
            len(frames_found)
            == len(requested_set)
        ):

            return frames_found, {
                "decoder":
                    "FFmpeg_sequential",
                "error": None
            }

        stderr_text = (
            stderr_data.decode(
                "utf-8",
                errors="replace"
            )
            if isinstance(
                stderr_data,
                bytes
            )
            else str(stderr_data)
        )

        return frames_found, {
            "decoder":
                "FFmpeg_sequential_partial",
            "error":
                stderr_text.strip()
                or
                (
                    "FFmpeg ended before "
                    "all requested frames "
                    "were decoded"
                )
        }

    except subprocess.TimeoutExpired:

        process.kill()

        process.communicate()

        return frames_found, {
            "decoder":
                "FFmpeg_timeout",
            "error":
                "FFmpeg decoding timeout"
        }

    except Exception as exc:

        return frames_found, {
            "decoder":
                "FFmpeg_exception",
            "error":
                str(exc)
        }


# =============================================================================
# 8. LOAD MASTER MANIFEST
# =============================================================================

print("\n" + "=" * 80)
print("LOADING STAGE 5.1 MASTER MANIFEST")
print("=" * 80)

if not MASTER_MANIFEST.exists():

    raise FileNotFoundError(
        MASTER_MANIFEST
    )

manifest = pd.read_csv(
    MASTER_MANIFEST
)

print(
    f"Videos in manifest: "
    f"{len(manifest):,}"
)

if "uid" not in manifest.columns:

    raise RuntimeError(
        "Master manifest does not contain uid."
    )

if "video_path" not in manifest.columns:

    raise RuntimeError(
        "Master manifest does not contain video_path."
    )

manifest = (
    manifest
    .drop_duplicates(
        "uid"
    )
    .reset_index(
        drop=True
    )
)

print(
    f"Unique videos: "
    f"{manifest['uid'].nunique():,}"
)


# =============================================================================
# 9. VALIDATE VIDEO PATHS
# =============================================================================

manifest["video_path"] = (
    manifest["video_path"]
    .astype(str)
    .str.strip()
)

manifest["_video_exists"] = (
    manifest["video_path"]
    .apply(
        lambda x:
            Path(x).exists()
    )
)

missing_video_count = int(
    (~manifest["_video_exists"])
    .sum()
)

print(
    f"Video files found: "
    f"{manifest['_video_exists'].sum():,}"
)

print(
    f"Video files missing: "
    f"{missing_video_count:,}"
)

if missing_video_count:

    print(
        "\nFirst missing video paths:"
    )

    print(
        manifest[
            ~manifest["_video_exists"]
        ][
            [
                "uid",
                "video_path"
            ]
        ]
        .head(20)
        .to_string(
            index=False
        )
    )

    raise RuntimeError(
        "\nStage 5.3 cannot proceed because "
        "the Stage 5.1 master manifest contains "
        "missing video files."
    )


# =============================================================================
# 10. DETERMINE EXACT 8 FRAME INDICES
# =============================================================================
#
# We reproduce the Stage 5.1 sampling rule:
#
#   np.linspace(
#       0,
#       frame_count - 1,
#       8
#   ).astype(int)
#
# This means the exact frame indices are determined
# from the original video metadata.
# =============================================================================

print("\n" + "=" * 80)
print("FRAME SAMPLING")
print("=" * 80)

all_frame_records = []

for row_no, row in manifest.iterrows():

    uid = str(
        row["uid"]
    )

    video_path = Path(
        row["video_path"]
    )

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Could not open video:\n"
            f"{video_path}"
        )

    frame_count = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    fps = float(
        cap.get(
            cv2.CAP_PROP_FPS
        )
    )

    width = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    height = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )

    cap.release()

    if frame_count <= 0:

        raise RuntimeError(
            f"Invalid frame count for:\n"
            f"{video_path}"
        )

    sample_indices = np.linspace(
        0,
        frame_count - 1,
        EXPECTED_FRAMES_PER_VIDEO
    ).astype(
        int
    )

    for sample_number, frame_index in enumerate(
        sample_indices
    ):

        all_frame_records.append({

            "uid": uid,

            "gloss": row.get(
                "gloss",
                ""
            ),

            "sample_number":
                int(sample_number),

            "frame_index":
                int(frame_index),

            "video_path":
                str(video_path),

            "frame_count":
                frame_count,

            "fps":
                fps,

            "width":
                width,

            "height":
                height
        })

    if (
        (row_no + 1) % 500 == 0
        or
        (row_no + 1) == len(manifest)
    ):

        print(
            f"[{row_no + 1:>5}/"
            f"{len(manifest)}] "
            f"sampling manifest generated"
        )


frame_plan = pd.DataFrame(
    all_frame_records
)

print(
    f"\nExpected frame records: "
    f"{len(frame_plan):,}"
)

assert (
    len(frame_plan)
    ==
    len(manifest)
    *
    EXPECTED_FRAMES_PER_VIDEO
)


# =============================================================================
# 11. INITIALIZE AUDIT
# =============================================================================

frame_audit_records = []

for _, row in frame_plan.iterrows():

    frame_audit_records.append({

        "uid":
            row["uid"],

        "gloss":
            row["gloss"],

        "sample_number":
            row["sample_number"],

        "frame_index":
            row["frame_index"],

        "video_path":
            row["video_path"],

        "frame_count":
            row["frame_count"],

        "fps":
            row["fps"],

        "width":
            row["width"],

        "height":
            row["height"],

        "frame_read":
            False,

        "decoder":
            "",

        "detector":
            DETECTOR_NAME,

        "detection_valid":
            False,

        "fallback_type":
            "",

        "num_detected_contours":
            0,

        "selected_num_hands":
            0,

        "confidence_proxy":
            0.0,

        "bbox_x1":
            np.nan,

        "bbox_y1":
            np.nan,

        "bbox_x2":
            np.nan,

        "bbox_y2":
            np.nan,

        "bbox_area_fraction":
            np.nan,

        "crop_x1":
            np.nan,

        "crop_y1":
            np.nan,

        "crop_x2":
            np.nan,

        "crop_y2":
            np.nan,

        "roi_path":
            "",

        "roi_saved":
            False,

        "latency_ms":
            np.nan,

        "error":
            ""
    })


frames = pd.DataFrame(
    frame_audit_records
)


# =============================================================================
# 12. PROCESS EACH VIDEO
# =============================================================================

print("\n" + "=" * 80)
print("ROI EXTRACTION")
print("=" * 80)

overall_start = time.time()

video_summaries = []

errors = []

total_videos = len(
    manifest
)

for video_no, (_, video_row) in enumerate(
    manifest.iterrows(),
    start=1
):

    uid = str(
        video_row["uid"]
    )

    gloss = video_row.get(
        "gloss",
        ""
    )

    video_path = Path(
        video_row["video_path"]
    )

    video_start = time.time()

    target_rows = frame_plan[
        frame_plan["uid"].astype(str)
        == uid
    ].copy()

    requested_indices = (
        target_rows[
            "frame_index"
        ]
        .astype(int)
        .tolist()
    )

    # -------------------------------------------------------------------------
    # Decode exact requested frames
    # -------------------------------------------------------------------------

    decoded_frames, decoder_info = (
        decode_requested_frames(
            video_path,
            requested_indices
        )
    )

    decoder_used = (
        decoder_info["decoder"]
    )

    decoder_error = (
        decoder_info["error"]
    )

    # -------------------------------------------------------------------------
    # Per-video state
    # -------------------------------------------------------------------------

    detected_count = 0

    temporal_fallback_count = 0

    full_frame_fallback_count = 0

    frame_read_count = 0

    saved_count = 0

    previous_roi = None

    video_errors = 0

    video_latencies = []

    video_area_fractions = []

    video_confidences = []

    # -------------------------------------------------------------------------
    # Process the exact requested frames in sample order
    # -------------------------------------------------------------------------

    for _, target in target_rows.sort_values(
        "sample_number"
    ).iterrows():

        sample_number = int(
            target[
                "sample_number"
            ]
        )

        frame_index = int(
            target[
                "frame_index"
            ]
        )

        frame_mask = (
            (frames["uid"].astype(str) == uid)
            &
            (
                frames["sample_number"]
                == sample_number
            )
            &
            (
                frames["frame_index"]
                == frame_index
            )
        )

        frame = decoded_frames.get(
            frame_index
        )

        if frame is None:

            error_text = (
                "Exact requested frame "
                "could not be decoded"
            )

            if decoder_error:

                error_text += (
                    " | "
                    + str(
                        decoder_error
                    )
                )

            frames.loc[
                frame_mask,
                "decoder"
            ] = decoder_used

            frames.loc[
                frame_mask,
                "error"
            ] = error_text

            errors.append({

                "uid":
                    uid,

                "frame_index":
                    frame_index,

                "sample_number":
                    sample_number,

                "decoder":
                    decoder_used,

                "error":
                    error_text
            })

            video_errors += 1

            continue

        frame_read_count += 1

        frames.loc[
            frame_mask,
            "frame_read"
        ] = True

        frames.loc[
            frame_mask,
            "decoder"
        ] = decoder_used

        # ---------------------------------------------------------------------
        # Detector
        # ---------------------------------------------------------------------

        t0 = time.perf_counter()

        roi_box, confidence_proxy, num_contours = (
            detect_opencv_skin_contour(
                frame
            )
        )

        detection_valid = (
            roi_box is not None
        )

        crop = None

        crop_box = None

        fallback_type = "none"

        # ---------------------------------------------------------------------
        # Detection succeeded
        # ---------------------------------------------------------------------

        if detection_valid:

            crop, crop_box = (
                crop_square_with_padding(
                    frame,
                    roi_box
                )
            )

            if crop is not None:

                previous_roi = (
                    roi_box
                )

                detected_count += 1

            else:

                detection_valid = False

        # ---------------------------------------------------------------------
        # Temporal fallback
        # ---------------------------------------------------------------------

        if crop is None:

            if previous_roi is not None:

                crop, crop_box = (
                    crop_square_with_padding(
                        frame,
                        previous_roi
                    )
                )

                if crop is not None:

                    roi_box = previous_roi

                    fallback_type = (
                        "temporal_fallback"
                    )

                    temporal_fallback_count += 1

        # ---------------------------------------------------------------------
        # Full-frame fallback
        # ---------------------------------------------------------------------

        if crop is None:

            crop = cv2.resize(
                frame,
                (
                    OUTPUT_SIZE,
                    OUTPUT_SIZE
                ),
                interpolation=cv2.INTER_AREA
            )

            fallback_type = (
                "full_frame_fallback"
            )

            full_frame_fallback_count += 1

            roi_box = (
                0,
                0,
                frame.shape[1],
                frame.shape[0]
            )

            crop_box = (
                0,
                0,
                frame.shape[1],
                frame.shape[0]
            )

        latency_ms = (
            time.perf_counter()
            - t0
        ) * 1000.0

        video_latencies.append(
            latency_ms
        )

        # ---------------------------------------------------------------------
        # ROI statistics
        # ---------------------------------------------------------------------

        x1, y1, x2, y2 = (
            roi_box
        )

        frame_area = float(
            frame.shape[0]
            *
            frame.shape[1]
        )

        bbox_area_fraction = (
            float(
                max(
                    0,
                    x2 - x1
                )
                *
                max(
                    0,
                    y2 - y1
                )
            )
            /
            frame_area
        )

        video_area_fractions.append(
            bbox_area_fraction
        )

        video_confidences.append(
            confidence_proxy
        )

        cx1, cy1, cx2, cy2 = (
            crop_box
        )

        # ---------------------------------------------------------------------
        # Output path
        # ---------------------------------------------------------------------

        output_dir = (
            ROI_DATASET
            /
            safe_uid_folder(uid)
        )

        output_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        output_path = (
            output_dir
            /
            f"sample_{sample_number:02d}.jpg"
        )

        write_ok = cv2.imwrite(
            str(output_path),
            crop,
            [
                cv2.IMWRITE_JPEG_QUALITY,
                JPEG_QUALITY
            ]
        )

        if (
            not write_ok
            or
            not output_path.exists()
        ):

            error_text = (
                "ROI image write failed"
            )

            frames.loc[
                frame_mask,
                "error"
            ] = error_text

            errors.append({

                "uid":
                    uid,

                "frame_index":
                    frame_index,

                "sample_number":
                    sample_number,

                "decoder":
                    decoder_used,

                "error":
                    error_text
            })

            video_errors += 1

            continue

        saved_count += 1

        # ---------------------------------------------------------------------
        # Update frame audit
        # ---------------------------------------------------------------------

        frames.loc[
            frame_mask,
            "detector"
        ] = DETECTOR_NAME

        frames.loc[
            frame_mask,
            "detection_valid"
        ] = detection_valid

        frames.loc[
            frame_mask,
            "fallback_type"
        ] = fallback_type

        frames.loc[
            frame_mask,
            "num_detected_contours"
        ] = num_contours

        frames.loc[
            frame_mask,
            "selected_num_hands"
        ] = num_contours

        frames.loc[
            frame_mask,
            "confidence_proxy"
        ] = confidence_proxy

        frames.loc[
            frame_mask,
            "bbox_x1"
        ] = x1

        frames.loc[
            frame_mask,
            "bbox_y1"
        ] = y1

        frames.loc[
            frame_mask,
            "bbox_x2"
        ] = x2

        frames.loc[
            frame_mask,
            "bbox_y2"
        ] = y2

        frames.loc[
            frame_mask,
            "bbox_area_fraction"
        ] = bbox_area_fraction

        frames.loc[
            frame_mask,
            "crop_x1"
        ] = cx1

        frames.loc[
            frame_mask,
            "crop_y1"
        ] = cy1

        frames.loc[
            frame_mask,
            "crop_x2"
        ] = cx2

        frames.loc[
            frame_mask,
            "crop_y2"
        ] = cy2

        frames.loc[
            frame_mask,
            "roi_path"
        ] = str(
            output_path
        )

        frames.loc[
            frame_mask,
            "roi_saved"
        ] = True

        frames.loc[
            frame_mask,
            "latency_ms"
        ] = latency_ms

        frames.loc[
            frame_mask,
            "error"
        ] = ""

    # -------------------------------------------------------------------------
    # Video summary
    # -------------------------------------------------------------------------

    frames_requested = (
        len(target_rows)
    )

    processing_time = (
        time.time()
        - video_start
    )

    video_summaries.append({

        "uid":
            uid,

        "gloss":
            gloss,

        "video_path":
            str(video_path),

        "frame_count":
            int(
                target_rows[
                    "frame_count"
                ].iloc[0]
            ),

        "fps":
            float(
                target_rows[
                    "fps"
                ].iloc[0]
            ),

        "width":
            int(
                target_rows[
                    "width"
                ].iloc[0]
            ),

        "height":
            int(
                target_rows[
                    "height"
                ].iloc[0]
            ),

        "frames_requested":
            frames_requested,

        "frames_processed":
            frame_read_count,

        "frames_saved":
            saved_count,

        "frame_read_failures":
            frames_requested
            - frame_read_count,

        "detected_roi_count":
            detected_count,

        "previous_roi_fallback_count":
            temporal_fallback_count,

        "full_frame_fallback_count":
            full_frame_fallback_count,

        "valid_detection_rate":
            (
                detected_count
                /
                frames_requested
            ),

        "fallback_rate":
            (
                (
                    temporal_fallback_count
                    +
                    full_frame_fallback_count
                )
                /
                frames_requested
            ),

        "mean_roi_area_fraction":
            (
                float(
                    np.mean(
                        video_area_fractions
                    )
                )
                if video_area_fractions
                else np.nan
            ),

        "mean_confidence_proxy":
            (
                float(
                    np.mean(
                        video_confidences
                    )
                )
                if video_confidences
                else np.nan
            ),

        "mean_latency_ms":
            (
                float(
                    np.mean(
                        video_latencies
                    )
                )
                if video_latencies
                else np.nan
            ),

        "total_processing_time_s":
            processing_time,

        "decoder":
            decoder_used,

        "decoder_error":
            (
                str(
                    decoder_error
                )
                if decoder_error
                else ""
            ),

        "status":
            (
                "COMPLETE"
                if (
                    saved_count
                    ==
                    frames_requested
                )
                else "INCOMPLETE"
            )
    })

    if (
        video_no % 100 == 0
        or
        video_no == total_videos
    ):

        # ---------------------------------------------------------------------
        # Progress report
        # ---------------------------------------------------------------------

        if (
            video_no % 100 == 0
            or video_no == total_videos
        ):

            total_saved_so_far = sum(
                item["frames_saved"]
                for item in video_summaries
            )

            print(
                f"[{video_no:>5}/{total_videos}] "
                f"saved={total_saved_so_far:,} | "
                f"errors={len(errors):,}"
            )

# =============================================================================
# 13. SAVE AUDITS
# =============================================================================

video_audit = pd.DataFrame(
    video_summaries
)

error_audit = pd.DataFrame(
    errors
)

frames.to_csv(
    FRAME_AUDIT,
    index=False
)

video_audit.to_csv(
    VIDEO_AUDIT,
    index=False
)

error_audit.to_csv(
    ERROR_AUDIT,
    index=False
)


# =============================================================================
# 14. FINAL DISK VERIFICATION
# =============================================================================

print("\n" + "=" * 80)
print("FINAL DISK VERIFICATION")
print("=" * 80)

expected_frames = len(
    frames
)

roi_files = list(
    ROI_DATASET.rglob(
        "*.jpg"
    )
)

saved_paths = (
    frames[
        "roi_path"
    ]
    .dropna()
    .astype(str)
)

existing_saved_paths = [
    p
    for p in saved_paths
    if Path(p).exists()
]

missing_on_disk = (
    expected_frames
    -
    len(existing_saved_paths)
)

duplicate_paths = int(
    saved_paths.duplicated().sum()
)


# =============================================================================
# 15. IMAGE DIMENSION VALIDATION
# =============================================================================

dimension_errors = []

for image_path in roi_files:

    image = cv2.imread(
        str(image_path)
    )

    if image is None:

        dimension_errors.append({
            "path":
                str(image_path),
            "error":
                "Unreadable image"
        })

        continue

    if (
        image.shape[0]
        != OUTPUT_SIZE
        or
        image.shape[1]
        != OUTPUT_SIZE
    ):

        dimension_errors.append({
            "path":
                str(image_path),
            "error":
                (
                    f"Expected "
                    f"{OUTPUT_SIZE}x"
                    f"{OUTPUT_SIZE}, "
                    f"got "
                    f"{image.shape[1]}x"
                    f"{image.shape[0]}"
                )
        })


# =============================================================================
# 16. VIDEO COMPLETENESS
# =============================================================================

video_save_counts = (
    frames.assign(
        _exists=
        frames[
            "roi_path"
        ].apply(
            lambda p:
                Path(str(p)).exists()
                if pd.notna(p)
                and str(p).strip()
                else False
        )
    )
    .groupby("uid")[
        "_exists"
    ]
    .sum()
)

complete_videos = int(
    (
        video_save_counts
        ==
        EXPECTED_FRAMES_PER_VIDEO
    ).sum()
)

incomplete_videos = int(
    len(video_save_counts)
    -
    complete_videos
)


# =============================================================================
# 17. METRICS
# =============================================================================

successful_detection_frames = int(
    frames[
        "detection_valid"
    ].fillna(False).sum()
)

valid_detection_rate = (
    successful_detection_frames
    /
    expected_frames
)

fallback_frames = int(
    frames[
        "fallback_type"
    ]
    .fillna("")
    .isin([
        "temporal_fallback",
        "full_frame_fallback"
    ])
    .sum()
)

fallback_rate = (
    fallback_frames
    /
    expected_frames
)

mean_roi_area = float(
    frames[
        "bbox_area_fraction"
    ]
    .dropna()
    .mean()
)

mean_confidence = float(
    frames[
        "confidence_proxy"
    ]
    .dropna()
    .mean()
)

mean_latency = float(
    frames[
        "latency_ms"
    ]
    .dropna()
    .mean()
)


# =============================================================================
# 18. STATUS
# =============================================================================

all_expected_images_created = (
    len(existing_saved_paths)
    ==
    expected_frames
)

all_videos_complete = (
    complete_videos
    ==
    len(video_save_counts)
)

no_dimension_errors = (
    len(dimension_errors)
    ==
    0
)

no_duplicate_paths = (
    duplicate_paths
    ==
    0
)

no_extraction_errors = (
    len(error_audit)
    ==
    0
)

stage53_pass = (
    all_expected_images_created
    and
    all_videos_complete
    and
    no_dimension_errors
    and
    no_duplicate_paths
    and
    no_extraction_errors
)


# =============================================================================
# 19. PROTOCOL FILE
# =============================================================================

protocol = {

    "stage":
        "5.3",

    "name":
        "Hand ROI Extraction",

    "dataset":
        "CISLR v1.5-a",

    "source_manifest":
        str(MASTER_MANIFEST),

    "videos":
        int(len(manifest)),

    "frames_per_video":
        EXPECTED_FRAMES_PER_VIDEO,

    "total_expected_frames":
        int(expected_frames),

    "input_resolution":
        "original video resolution",

    "output_resolution":
        "224x224",

    "detector":
        DETECTOR_NAME,

    "detector_parameters": {

        "HSV_lower":
            HSV_LOWER.tolist(),

        "HSV_upper":
            HSV_UPPER.tolist(),

        "YCrCb_lower":
            YCRCB_LOWER.tolist(),

        "YCrCb_upper":
            YCRCB_UPPER.tolist(),

        "morphology_kernel":
            MORPH_KERNEL_SIZE,

        "minimum_contour_area":
            MIN_CONTOUR_AREA,

        "minimum_roi_area_fraction":
            ROI_MIN_AREA_FRACTION,

        "maximum_roi_area_fraction":
            ROI_MAX_AREA_FRACTION,

        "crop_padding_factor":
            CROP_PADDING_FACTOR
    },

    "decoding_policy": {

        "primary":
            "OpenCV sequential decoding",

        "fallback":
            "FFmpeg sequential decoding",

        "random_frame_seeking":
            False,

        "neighbor_frame_substitution":
            False,

        "exact_frame_index_required":
            True
    },

    "roi_policy": {

        "detected":
            "Current frame valid detector ROI",

        "temporal_fallback":
            "Previous valid ROI within same video",

        "full_frame_fallback":
            "Full frame only when no previous valid ROI exists"
    },

    "jpeg_quality":
        JPEG_QUALITY,

    "training_performed":
        False,

    "retrieval_performed":
        False,

    "raw_videos_modified":
        False
}

with open(
    PROTOCOL_FILE,
    "w"
) as f:

    json.dump(
        protocol,
        f,
        indent=2
    )


# =============================================================================
# 20. SUMMARY
# =============================================================================

summary = {

    "stage":
        "5.3",

    "status":
        (
            "PASS"
            if stage53_pass
            else "FAIL"
        ),

    "videos":
        int(len(manifest)),

    "frames_expected":
        int(expected_frames),

    "frames_saved":
        int(
            len(existing_saved_paths)
        ),

    "missing_images":
        int(missing_on_disk),

    "complete_videos":
        int(complete_videos),

    "incomplete_videos":
        int(incomplete_videos),

    "valid_detection_rate":
        float(valid_detection_rate),

    "fallback_rate":
        float(fallback_rate),

    "mean_roi_area_fraction":
        mean_roi_area,

    "mean_confidence_proxy":
        mean_confidence,

    "mean_extraction_latency_ms":
        mean_latency,

    "roi_jpg_files":
        int(len(roi_files)),

    "dimension_errors":
        int(len(dimension_errors)),

    "duplicate_output_paths":
        duplicate_paths,

    "extraction_errors":
        int(len(error_audit)),

    "decoder_counts":
        (
            frames[
                "decoder"
            ]
            .value_counts(
                dropna=False
            )
            .to_dict()
        ),

    "raw_videos_modified":
        False,

    "training_performed":
        False,

    "retrieval_performed":
        False
}

with open(
    SUMMARY_FILE,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# =============================================================================
# 21. HASH AUDIT
# =============================================================================

hash_audit = {

    "master_manifest_sha256":
        sha256_file(
            MASTER_MANIFEST
        ),

    "protocol_sha256":
        sha256_file(
            PROTOCOL_FILE
        ),

    "frame_audit_sha256":
        sha256_file(
            FRAME_AUDIT
        ),

    "video_audit_sha256":
        sha256_file(
            VIDEO_AUDIT
        ),

    "error_audit_sha256":
        (
            sha256_file(
                ERROR_AUDIT
            )
            if ERROR_AUDIT.exists()
            else None
        ),

    "summary_sha256":
        sha256_file(
            SUMMARY_FILE
        )
}

with open(
    HASH_FILE,
    "w"
) as f:

    json.dump(
        hash_audit,
        f,
        indent=2
    )


# =============================================================================
# 22. FINAL REPORT
# =============================================================================

print("\n" + "=" * 80)
print("STAGE 5.3 — FINAL RESULT")
print("=" * 80)

print("\nDATASET")
print("-" * 80)

print(
    f"Videos                       : "
    f"{len(manifest):,}"
)

print(
    f"Expected frames              : "
    f"{expected_frames:,}"
)

print(
    f"Saved ROI images             : "
    f"{len(existing_saved_paths):,}"
)

print(
    f"Missing ROI images           : "
    f"{missing_on_disk:,}"
)

print(
    f"ROI JPG files on disk        : "
    f"{len(roi_files):,}"
)

print(
    f"Complete videos              : "
    f"{complete_videos:,}"
)

print(
    f"Incomplete videos            : "
    f"{incomplete_videos:,}"
)

print("\nROI QUALITY")
print("-" * 80)

print(
    f"Valid detection rate         : "
    f"{valid_detection_rate:.4f}"
)

print(
    f"Fallback rate                : "
    f"{fallback_rate:.4f}"
)

print(
    f"Mean ROI area fraction       : "
    f"{mean_roi_area:.4f}"
)

print(
    f"Mean confidence proxy       : "
    f"{mean_confidence:.6f}"
)

print(
    f"Mean extraction latency     : "
    f"{mean_latency:.3f} ms/frame"
)

print("\nDECODING")
print("-" * 80)

print(
    frames[
        "decoder"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)

print("\nVALIDATION")
print("-" * 80)

print(
    "All expected ROI images created : "
    +
    (
        "PASS"
        if all_expected_images_created
        else "FAIL"
    )
)

print(
    "All videos have 8 saved frames   : "
    +
    (
        "PASS"
        if all_videos_complete
        else "FAIL"
    )
)

print(
    "All images are 224x224           : "
    +
    (
        "PASS"
        if no_dimension_errors
        else "FAIL"
    )
)

print(
    "No duplicate output paths        : "
    +
    (
        "PASS"
        if no_duplicate_paths
        else "FAIL"
    )
)

print(
    "No extraction errors             : "
    +
    (
        "PASS"
        if no_extraction_errors
        else "FAIL"
    )
)

print(
    "Raw videos modified              : NO"
)

print(
    "Training performed               : NO"
)

print(
    "Retrieval performed              : NO"
)

print("\n" + "=" * 80)

if stage53_pass:

    print(
        "✅ STAGE 5.3 PASSED"
    )

    print(
        "\nThe ROI dataset is complete."
    )

    print(
        "Next: STAGE 5.4 — ROI QUALITY VALIDATION."
    )

else:

    print(
        "⚠️ STAGE 5.3 DID NOT PASS"
    )

    print(
        "\nDo NOT start Stage 5.4."
    )

    print(
        f"Remaining error audit:\n"
        f"{ERROR_AUDIT}"
    )

print("\nArtifacts:")
print(
    f"  Frame audit : {FRAME_AUDIT}"
)

print(
    f"  Video audit : {VIDEO_AUDIT}"
)

print(
    f"  Error audit : {ERROR_AUDIT}"
)

print(
    f"  Summary     : {SUMMARY_FILE}"
)

print(
    f"  Protocol    : {PROTOCOL_FILE}"
)

print(
    f"  Hash audit  : {HASH_FILE}"
)

print("=" * 80)

STAGE 5.3 — CLEAN RESTART

The following previous Stage 5.3 outputs will be removed:
  /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/roi_extraction
  /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/roi_dataset

Stage 5.1 and Stage 5.2 will NOT be touched.
Raw CISLR videos will NOT be touched.
Stage 4 artifacts will NOT be touched.

Removing: /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/roi_extraction

Removing: /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/roi_dataset

DECODER AVAILABILITY
ffmpeg : /usr/bin/ffmpeg
ffprobe: /usr/bin/ffprobe

LOADING STAGE 5.1 MASTER MANIFEST
Videos in manifest: 7,049
Unique videos: 7,049
Video files found: 7,049
Video files missing: 0

FRAME SAMPLING
[  500/7049] sampling manifest generated
[ 1000/7049] sampling manifest generated
[ 1500/7049] sampling manifest generated
[ 2000/7049] sampling manifest generated
[ 2500/7049] sampling manifest generated
[ 3000/7049] sampling manifest gener

In [8]:
# =============================================================================
# STAGE 5.3 — ERROR-TOLERANT MPEG-4 RECOVERY DIAGNOSTIC
# =============================================================================
#
# PURPOSE:
#   Test whether FFmpeg can continue decoding through the damaged MPEG-4
#   region using -err_detect ignore_err.
#
# IMPORTANT:
#   - Diagnostic ONLY
#   - Does NOT modify raw videos
#   - Does NOT modify ROI dataset
#   - Does NOT create final ROI images
#   - Does NOT substitute neighboring frames
#
# SCIENTIFIC RULE:
#   A recovery is acceptable ONLY if the requested frame index itself can be
#   decoded. If FFmpeg merely skips the damaged frame, that is NOT recovery.
# =============================================================================

from pathlib import Path
import pandas as pd
import subprocess
import json
import os
import tempfile
import shutil

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

ROI_EXTRACTION_DIR = (
    PROJECT_ROOT
    / "audit"
    / "stage5_hand_roi"
    / "roi_extraction"
)

DIAGNOSTIC_DIR = (
    ROI_EXTRACTION_DIR
    / "decoder_diagnostic"
)

INPUT_CSV = (
    DIAGNOSTIC_DIR
    / "stage5_3_problematic_video_diagnostic.csv"
)

OUTPUT_CSV = (
    DIAGNOSTIC_DIR
    / "stage5_3_error_tolerant_recovery_diagnostic.csv"
)

# =============================================================================
# LOAD PREVIOUS DIAGNOSTIC
# =============================================================================

diag = pd.read_csv(INPUT_CSV)

print("=" * 80)
print("STAGE 5.3 — ERROR-TOLERANT MPEG-4 RECOVERY DIAGNOSTIC")
print("=" * 80)

print(
    f"\nRepresentative problematic videos : {len(diag)}"
)

# =============================================================================
# HELPER
# =============================================================================

def run_ffmpeg_ignore_errors(video_path, output_dir):

    """
    Decode the entire video using FFmpeg while asking the decoder
    to continue past recoverable decoding errors.

    Each successfully decoded frame is written as an individual PNG.

    This is diagnostic only.
    """

    output_pattern = (
        str(output_dir / "frame_%06d.png")
    )

    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "warning",

        # Continue through decoding errors when possible
        "-err_detect", "ignore_err",

        "-i", str(video_path),

        # Decode every frame that FFmpeg can recover
        "-vsync", "0",

        "-f", "image2",
        output_pattern,
    ]

    proc = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        timeout=120,
        check=False,
    )

    stderr = proc.stderr.decode(
        "utf-8",
        errors="replace"
    )

    frames = sorted(
        output_dir.glob(
            "frame_*.png"
        )
    )

    return proc.returncode, frames, stderr


# =============================================================================
# RUN
# =============================================================================

results = []

for idx, row in diag.iterrows():

    uid = str(row["uid"])

    video_path = Path(
        str(row["video_path"])
    )

    target_frame = int(
        row["target_frame"]
    )

    print(
        f"\n[{idx+1}/{len(diag)}] "
        f"{uid} | target frame={target_frame}"
    )

    # -------------------------------------------------------------
    # Temporary directory
    # -------------------------------------------------------------

    temp_dir = Path(
        tempfile.mkdtemp(
            prefix="parasurg_ffmpeg_recovery_",
            dir=str(DIAGNOSTIC_DIR)
        )
    )

    try:

        return_code, frames, stderr = (
            run_ffmpeg_ignore_errors(
                video_path,
                temp_dir
            )
        )

        decoded_count = len(frames)

        # ---------------------------------------------------------
        # FFmpeg image2 output is sequentially numbered starting
        # from 1.
        #
        # Therefore:
        #
        # requested frame index N
        # corresponds to output frame N+1
        #
        # ONLY if FFmpeg decoded every preceding frame.
        # ---------------------------------------------------------

        expected_output_number = (
            target_frame + 1
        )

        expected_output = (
            temp_dir
            / f"frame_{expected_output_number:06d}.png"
        )

        exact_frame_file_exists = (
            expected_output.exists()
        )

        # ---------------------------------------------------------
        # Determine whether FFmpeg produced enough frames
        # ---------------------------------------------------------

        enough_frames = (
            decoded_count
            > target_frame
        )

        # ---------------------------------------------------------
        # IMPORTANT:
        # If FFmpeg skipped a damaged frame, simply having N+1
        # output files does NOT prove that output N corresponds
        # to source frame N.
        #
        # We therefore record this only as "candidate recovery",
        # not accepted recovery.
        # ---------------------------------------------------------

        if (
            exact_frame_file_exists
            and enough_frames
        ):

            candidate_status = (
                "CANDIDATE_EXACT_POSITION"
            )

        else:

            candidate_status = (
                "NOT_RECOVERED"
            )

        # ---------------------------------------------------------
        # Capture relevant decoder errors
        # ---------------------------------------------------------

        error_lines = []

        for line in stderr.splitlines():

            lower = line.lower()

            if any(
                keyword in lower
                for keyword in [
                    "error",
                    "invalid",
                    "corrupt",
                    "concealing",
                    "missing",
                    "decode",
                    "damaged",
                    "timestamp",
                ]
            ):

                error_lines.append(
                    line.strip()
                )

        error_text = "\n".join(
            error_lines[:20]
        )

        results.append({

            "uid": uid,

            "target_frame": target_frame,

            "video_path": str(
                video_path
            ),

            "ffmpeg_return_code": (
                return_code
            ),

            "ignore_error_decoded_frames": (
                decoded_count
            ),

            "target_plus_one_required": (
                target_frame + 1
            ),

            "expected_output_file_exists": (
                exact_frame_file_exists
            ),

            "enough_decoded_frames": (
                enough_frames
            ),

            "candidate_status": (
                candidate_status
            ),

            "decoder_error_excerpt": (
                error_text
            ),
        })

        print(
            f"  decoded frames : {decoded_count}"
        )

        print(
            f"  required       : {target_frame + 1}"
        )

        print(
            f"  candidate      : {candidate_status}"
        )

    except subprocess.TimeoutExpired:

        results.append({

            "uid": uid,

            "target_frame": target_frame,

            "video_path": str(
                video_path
            ),

            "ffmpeg_return_code": None,

            "ignore_error_decoded_frames": None,

            "target_plus_one_required": (
                target_frame + 1
            ),

            "expected_output_file_exists": False,

            "enough_decoded_frames": False,

            "candidate_status": (
                "FFMPEG_TIMEOUT"
            ),

            "decoder_error_excerpt": (
                "FFmpeg timeout."
            ),
        })

        print(
            "  FFmpeg timeout"
        )

    finally:

        # Remove temporary diagnostic frames.
        # Nothing from this directory is used as final ROI data.
        shutil.rmtree(
            temp_dir,
            ignore_errors=True
        )

# =============================================================================
# SAVE
# =============================================================================

result_df = pd.DataFrame(
    results
)

result_df.to_csv(
    OUTPUT_CSV,
    index=False
)

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("ERROR-TOLERANT RECOVERY SUMMARY")
print("=" * 80)

print(
    "\nCandidate statuses:"
)

print(
    result_df[
        "candidate_status"
    ]
    .value_counts()
    .to_string()
)

print("\nDetailed results:")

display(
    result_df[
        [
            "uid",
            "target_frame",
            "ffmpeg_return_code",
            "ignore_error_decoded_frames",
            "target_plus_one_required",
            "expected_output_file_exists",
            "enough_decoded_frames",
            "candidate_status",
        ]
    ]
)

print("\nSaved:")
print(OUTPUT_CSV)

print("\n" + "=" * 80)
print("IMPORTANT")
print("=" * 80)

print(
    """
This is still diagnostic.

Do NOT use these frames for the ROI dataset yet.

A candidate recovery is NOT automatically accepted because
FFmpeg may skip corrupted frames. We must establish that the
requested source frame index is actually preserved.

If all candidates remain unrecovered, we have strong evidence
that the requested frames are genuinely undecodable.
"""
)

STAGE 5.3 — ERROR-TOLERANT MPEG-4 RECOVERY DIAGNOSTIC

Representative problematic videos : 9

[1/9] -0XujLX2_9I_2 | target frame=59
  decoded frames : 58
  required       : 60
  candidate      : NOT_RECOVERED

[2/9] -0XujLX2_9I_2 | target frame=69
  decoded frames : 58
  required       : 70
  candidate      : NOT_RECOVERED

[3/9] 7hICMJlI2hQ_2 | target frame=140
  decoded frames : 136
  required       : 141
  candidate      : NOT_RECOVERED

[4/9] GA00BzF9EpE_2 | target frame=103
  decoded frames : 94
  required       : 104
  candidate      : NOT_RECOVERED

[5/9] QKEuQ32GLNc_2 | target frame=114
  decoded frames : 105
  required       : 115
  candidate      : NOT_RECOVERED

[6/9] b8BlqEQCtiY_2 | target frame=141
  decoded frames : 136
  required       : 142
  candidate      : NOT_RECOVERED

[7/9] pr_5nYeLFKc_2 | target frame=92
  decoded frames : 88
  required       : 93
  candidate      : NOT_RECOVERED

[8/9] LYfzb9LaUHw_2 | target frame=99
  decoded frames : 91
  required       : 100


,uid,target_frame,ffmpeg_return_code,ignore_error_decoded_frames,target_plus_one_required,expected_output_file_exists,enough_decoded_frames,candidate_status
0,-0XujLX2_9I_2,59,0,58,60,False,False,NOT_RECOVERED
1,-0XujLX2_9I_2,69,0,58,70,False,False,NOT_RECOVERED
2,7hICMJlI2hQ_2,140,0,136,141,False,False,NOT_RECOVERED
3,GA00BzF9EpE_2,103,0,94,104,False,False,NOT_RECOVERED
4,QKEuQ32GLNc_2,114,0,105,115,False,False,NOT_RECOVERED
5,b8BlqEQCtiY_2,141,0,136,142,False,False,NOT_RECOVERED
6,pr_5nYeLFKc_2,92,0,88,93,False,False,NOT_RECOVERED
7,LYfzb9LaUHw_2,99,0,91,100,False,False,NOT_RECOVERED
8,zupZnldqNbE_2,164,0,160,165,False,False,NOT_RECOVERED



Saved:
/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/roi_extraction/decoder_diagnostic/stage5_3_error_tolerant_recovery_diagnostic.csv

IMPORTANT

This is still diagnostic.

Do NOT use these frames for the ROI dataset yet.

A candidate recovery is NOT automatically accepted because
FFmpeg may skip corrupted frames. We must establish that the
requested source frame index is actually preserved.

If all candidates remain unrecovered, we have strong evidence
that the requested frames are genuinely undecodable.



In [12]:
# ============================================================
# PARASURG / CISLR
# STAGE 5.3 — FINALIZATION & ANALYSIS-VALID SUBSET FREEZE
# ============================================================
#
# PURPOSE:
#   1. Load the ACTUAL Stage 5.3 extraction artifacts.
#   2. Identify videos with complete 8-frame ROI extraction.
#   3. Freeze the analysis-valid UID set.
#   4. Create a matched manifest for:
#        - Full Frame
#        - Hand ROI
#   5. Record the decoder-exclusion evidence.
#
# IMPORTANT:
#   - DO NOT rerun extraction.
#   - DO NOT rerun decoder diagnostics.
#   - DO NOT modify raw videos.
#   - DO NOT modify Stage 4.
#   - DO NOT modify Stage 5.1 or Stage 5.2.
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib
from datetime import datetime

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi"
)

EXTRACT_DIR = ROOT / "roi_extraction"
ROI_DATASET_DIR = ROOT / "roi_dataset"

FRAME_AUDIT = EXTRACT_DIR / "stage5_3_frame_level_extraction.csv"
VIDEO_AUDIT = EXTRACT_DIR / "stage5_3_video_level_summary.csv"
ERROR_AUDIT = EXTRACT_DIR / "stage5_3_extraction_errors.csv"

SOURCE_CHECK = (
    EXTRACT_DIR
    / "source_video_check"
    / "stage5_3_failed_source_video_check.csv"
)

TARGETED_REPAIR = (
    EXTRACT_DIR
    / "targeted_repair"
    / "stage5_3_targeted_repair_audit.csv"
)

OUTPUT_DIR = ROOT / "analysis_valid_subset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 90)
print("STAGE 5.3 — FINALIZATION & ANALYSIS-VALID SUBSET FREEZE")
print("=" * 90)

# ------------------------------------------------------------
# 2. VERIFY REQUIRED ARTIFACTS
# ------------------------------------------------------------

required = {
    "Frame-level extraction audit": FRAME_AUDIT,
    "Video-level extraction audit": VIDEO_AUDIT,
    "Extraction error audit": ERROR_AUDIT,
    "Failed-source-video diagnostic": SOURCE_CHECK,
}

print("\nArtifact verification:\n")

missing = []

for description, path in required.items():
    exists = path.exists()
    print(f"{'✓' if exists else '✗'} {description}")
    print(f"    {path}")

    if not exists:
        missing.append(path)

if missing:
    raise FileNotFoundError(
        "\nRequired Stage 5.3 artifacts are missing:\n"
        + "\n".join(str(p) for p in missing)
    )

print("\nAll required artifacts found.")

# ------------------------------------------------------------
# 3. LOAD ARTIFACTS
# ------------------------------------------------------------

frame_df = pd.read_csv(FRAME_AUDIT)
video_df = pd.read_csv(VIDEO_AUDIT)
error_df = pd.read_csv(ERROR_AUDIT)
source_df = pd.read_csv(SOURCE_CHECK)

print("\nLoaded artifact shapes:")
print("Frame audit :", frame_df.shape)
print("Video audit :", video_df.shape)
print("Error audit :", error_df.shape)
print("Source check:", source_df.shape)

# ------------------------------------------------------------
# 4. DISPLAY COLUMN NAMES
# ------------------------------------------------------------

print("\nFrame audit columns:")
print(list(frame_df.columns))

print("\nVideo audit columns:")
print(list(video_df.columns))

print("\nError audit columns:")
print(list(error_df.columns))

# ------------------------------------------------------------
# 5. IDENTIFY UID COLUMN
# ------------------------------------------------------------

def find_uid_column(df, name):
    candidates = [
        "uid",
        "video_id",
        "video_uid",
        "UID",
        "video",
    ]

    for c in candidates:
        if c in df.columns:
            print(f"{name}: using UID column -> {c}")
            return c

    raise KeyError(
        f"Could not identify UID column in {name}. "
        f"Available columns: {list(df.columns)}"
    )

frame_uid_col = find_uid_column(frame_df, "Frame audit")
video_uid_col = find_uid_column(video_df, "Video audit")

# ------------------------------------------------------------
# 6. NORMALIZE UID TYPES
# ------------------------------------------------------------

frame_df["_uid"] = frame_df[frame_uid_col].astype(str)
video_df["_uid"] = video_df[video_uid_col].astype(str)

# ------------------------------------------------------------
# 7. DETERMINE EXPECTED FRAME COUNT
# ------------------------------------------------------------

EXPECTED_FRAMES_PER_VIDEO = 8

# ------------------------------------------------------------
# 8. DETERMINE COMPLETE VIDEOS
# ------------------------------------------------------------

# Count successfully saved ROI images from frame-level audit.
#
# We intentionally use the frame-level audit because this is the
# most direct evidence of which prescribed frames actually produced
# ROI outputs.

possible_success_columns = [
    "roi_image_path",
    "output_path",
    "roi_path",
    "saved_path",
    "image_path",
]

success_col = None

for c in possible_success_columns:
    if c in frame_df.columns:
        success_col = c
        break

print("\nDetected output-path column:", success_col)

if success_col is not None:

    frame_df["_output_exists"] = (
        frame_df[success_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

else:
    # Fall back to an explicit success/status column if present.
    status_candidates = [
        "status",
        "extraction_status",
        "result",
    ]

    status_col = None

    for c in status_candidates:
        if c in frame_df.columns:
            status_col = c
            break

    if status_col is None:
        raise KeyError(
            "Could not identify an ROI output-path or status column.\n"
            f"Frame audit columns: {list(frame_df.columns)}"
        )

    print("Using status column:", status_col)

    frame_df["_output_exists"] = (
        frame_df[status_col]
        .astype(str)
        .str.lower()
        .isin([
            "success",
            "saved",
            "detected",
            "temporal_fallback",
            "full_frame_fallback",
        ])
    )

# ------------------------------------------------------------
# 9. VIDEO-LEVEL SUCCESS COUNTS
# ------------------------------------------------------------

video_counts = (
    frame_df
    .groupby("_uid")
    .agg(
        expected_frames=("_uid", "size"),
        successful_frames=("_output_exists", "sum"),
    )
    .reset_index()
)

video_counts["complete"] = (
    (video_counts["expected_frames"] == EXPECTED_FRAMES_PER_VIDEO)
    &
    (video_counts["successful_frames"] == EXPECTED_FRAMES_PER_VIDEO)
)

complete_uids = set(
    video_counts.loc[video_counts["complete"], "_uid"]
)

incomplete_uids = set(
    video_counts.loc[~video_counts["complete"], "_uid"]
)

print("\n" + "=" * 90)
print("VIDEO COMPLETENESS")
print("=" * 90)

print("Unique videos in frame audit :", len(video_counts))
print("Complete 8-frame videos      :", len(complete_uids))
print("Incomplete videos            :", len(incomplete_uids))

# ------------------------------------------------------------
# 10. EXPECTED CISLR MASTER SET
# ------------------------------------------------------------

EXPECTED_TOTAL_VIDEOS = 7049

print("\nExpected Stage 5.3 valid CISLR videos:", EXPECTED_TOTAL_VIDEOS)

if len(video_counts) != EXPECTED_TOTAL_VIDEOS:
    print(
        "WARNING: Frame audit UID count differs from the expected "
        f"{EXPECTED_TOTAL_VIDEOS} videos."
    )

# ------------------------------------------------------------
# 11. FAILED VIDEO EVIDENCE
# ------------------------------------------------------------

failed_uids_from_source = set()

if "_uid" not in source_df.columns:
    source_uid_col = None

    for c in ["uid", "video_id", "video_uid", "UID", "video"]:
        if c in source_df.columns:
            source_uid_col = c
            break

    if source_uid_col is not None:
        source_df["_uid"] = source_df[source_uid_col].astype(str)

if "_uid" in source_df.columns:
    failed_uids_from_source = set(source_df["_uid"])

print("\nDecoder/source diagnostic:")
print("Unique failed source videos:", len(failed_uids_from_source))

# ------------------------------------------------------------
# 12. CONSISTENCY CHECK
# ------------------------------------------------------------

overlap = complete_uids.intersection(failed_uids_from_source)

print("Complete videos also appearing in failed-source diagnostic:",
      len(overlap))

if len(overlap) > 0:
    print(
        "\nWARNING: Some videos appear both complete and in the "
        "failed-source diagnostic. This should be inspected."
    )

# ------------------------------------------------------------
# 13. FREEZE ANALYSIS-VALID UID SET
# ------------------------------------------------------------

analysis_valid_uids = sorted(complete_uids)

analysis_excluded_uids = sorted(incomplete_uids)

print("\n" + "=" * 90)
print("ANALYSIS-VALID SUBSET")
print("=" * 90)

print("Analysis-valid videos :", len(analysis_valid_uids))
print("Excluded videos       :", len(analysis_excluded_uids))

# ------------------------------------------------------------
# 14. CREATE FROZEN UID MANIFEST
# ------------------------------------------------------------

valid_manifest = pd.DataFrame({
    "uid": analysis_valid_uids,
    "analysis_status": "analysis_valid",
    "roi_frames_required": EXPECTED_FRAMES_PER_VIDEO,
    "roi_frames_available": EXPECTED_FRAMES_PER_VIDEO,
})

excluded_manifest = pd.DataFrame({
    "uid": analysis_excluded_uids,
    "analysis_status": "excluded_incomplete_roi",
})

valid_manifest_path = (
    OUTPUT_DIR / "stage5_3_analysis_valid_uid_manifest.csv"
)

excluded_manifest_path = (
    OUTPUT_DIR / "stage5_3_analysis_excluded_uid_manifest.csv"
)

valid_manifest.to_csv(valid_manifest_path, index=False)
excluded_manifest.to_csv(excluded_manifest_path, index=False)

# ------------------------------------------------------------
# 15. CREATE MATCHED FULL-FRAME vs ROI MANIFEST
# ------------------------------------------------------------

matched_manifest = valid_manifest.copy()

matched_manifest["full_frame_condition"] = "FULL_FRAME"
matched_manifest["roi_condition"] = "HAND_ROI"

matched_manifest_path = (
    OUTPUT_DIR / "stage5_3_matched_comparison_manifest.csv"
)

matched_manifest.to_csv(
    matched_manifest_path,
    index=False
)

# ------------------------------------------------------------
# 16. CREATE VIDEO COMPLETENESS AUDIT
# ------------------------------------------------------------

video_counts_out = video_counts.copy()

video_counts_out["analysis_status"] = video_counts_out["complete"].map({
    True: "analysis_valid",
    False: "excluded_incomplete_roi",
})

video_counts_path = (
    OUTPUT_DIR / "stage5_3_video_completeness_audit.csv"
)

video_counts_out.to_csv(
    video_counts_path,
    index=False
)

# ------------------------------------------------------------
# 17. CREATE EXCLUSION EVIDENCE
# ------------------------------------------------------------

exclusion_evidence = pd.DataFrame({
    "uid": analysis_excluded_uids
})

if len(exclusion_evidence) > 0:
    exclusion_evidence["reason"] = (
        "At least one prescribed Stage 5.3 sampled frame "
        "did not yield a saved ROI image after tested extraction "
        "and targeted repair procedures."
    )
else:
    exclusion_evidence["reason"] = []

exclusion_evidence_path = (
    OUTPUT_DIR / "stage5_3_exclusion_evidence.csv"
)

exclusion_evidence.to_csv(
    exclusion_evidence_path,
    index=False
)

# ------------------------------------------------------------
# 18. HASH THE FROZEN VALID UID MANIFEST
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()

manifest_hash = sha256_file(valid_manifest_path)

# ------------------------------------------------------------
# 19. FINALIZATION SUMMARY
# ------------------------------------------------------------

summary = {
    "stage": "5.3",
    "experiment": "Hand ROI Extraction",
    "timestamp": datetime.now().isoformat(),

    "dataset": "CISLR_v1.5-a",

    "expected_total_videos": EXPECTED_TOTAL_VIDEOS,

    "analysis_valid_videos": len(analysis_valid_uids),

    "excluded_incomplete_videos": len(
        analysis_excluded_uids
    ),

    "expected_frames_per_video": EXPECTED_FRAMES_PER_VIDEO,

    "expected_analysis_frames": (
        len(analysis_valid_uids)
        * EXPECTED_FRAMES_PER_VIDEO
    ),

    "comparison_design": (
        "Matched Full-Frame vs Hand-ROI comparison "
        "using identical analysis-valid UID set."
    ),

    "roi_detector": "OpenCV_Skin_Contour",

    "backbone": "MobileNetV3-Large",

    "embedding_dimension": 960,

    "retrieval_protocol": (
        "Official CISLR prototype-to-test retrieval "
        "with cosine similarity."
    ),

    "excluded_video_policy": (
        "Videos with incomplete prescribed ROI frame extraction "
        "are excluded from the controlled Stage 5 comparison."
    ),

    "scientific_exclusion_wording": (
        "Videos containing at least one prescribed sampled frame "
        "that was not decodable by the tested extraction methods, "
        "despite valid container-level metadata, are excluded "
        "from the matched analysis."
    ),

    "raw_videos_modified": False,

    "stage4_modified": False,

    "stage5_1_modified": False,

    "stage5_2_modified": False,

    "valid_manifest_sha256": manifest_hash,
}

summary_path = (
    OUTPUT_DIR / "stage5_3_finalization_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

# ------------------------------------------------------------
# 20. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("STAGE 5.3 FINALIZATION COMPLETE")
print("=" * 90)

print(f"""
Dataset:
    CISLR v1.5-a

Original valid videos:
    {EXPECTED_TOTAL_VIDEOS}

Analysis-valid videos:
    {len(analysis_valid_uids)}

Excluded incomplete videos:
    {len(analysis_excluded_uids)}

Frames/video:
    {EXPECTED_FRAMES_PER_VIDEO}

Analysis-valid ROI frames:
    {len(analysis_valid_uids) * EXPECTED_FRAMES_PER_VIDEO}

ROI detector:
    OpenCV_Skin_Contour

Backbone:
    MobileNetV3-Large

Embedding:
    960-D

Comparison:
    Matched Full-Frame vs Hand-ROI

Retrieval:
    Official prototype → test cosine retrieval

Raw videos modified:
    NO

Stage 4 modified:
    NO

Stage 5.1 modified:
    NO

Stage 5.2 modified:
    NO
""")

print("Artifacts written to:")
print(OUTPUT_DIR)

print("\nCreated files:")

for p in sorted(OUTPUT_DIR.glob("*")):
    print("  ", p.name)

print("\nFrozen valid UID manifest SHA-256:")
print(manifest_hash)

print("\n" + "=" * 90)
print("IMPORTANT")
print("=" * 90)

print("""
DO NOT rerun:
    - Stage 5.3 extraction
    - targeted repair
    - decoder diagnostics
    - Stage 5.1
    - Stage 5.2
    - Stage 4

The analysis-valid UID manifest created here will be the
common matched video set for Stage 5.5, Stage 5.6 and Stage 5.7.
""")

STAGE 5.3 — FINALIZATION & ANALYSIS-VALID SUBSET FREEZE

Artifact verification:

✓ Frame-level extraction audit
    /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/roi_extraction/stage5_3_frame_level_extraction.csv
✓ Video-level extraction audit
    /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/roi_extraction/stage5_3_video_level_summary.csv
✓ Extraction error audit
    /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/roi_extraction/stage5_3_extraction_errors.csv
✓ Failed-source-video diagnostic
    /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/roi_extraction/source_video_check/stage5_3_failed_source_video_check.csv

All required artifacts found.

Loaded artifact shapes:
Frame audit : (56392, 30)
Video audit : (7049, 23)
Error audit : (422, 5)
Source check: (421, 21)

Frame audit columns:
['uid', 'gloss', 'sample_number', 'frame_index', 'video_path', 'frame_count', 'fps', 'width', 'height', 'frame_read', 'decoder', 'detecto